# Artistic Telephone: live reveal

One chain at a time, **one large image at a time**. Keep the next images and their prompts secret until you decide to reveal them. After the fifth round, show a contact sheet with **all five images and all available prompts** for that chain. Then move to the next chain.

The presentation runs inside JupyterLab using the existing `ipywidgets` controls. Use full-screen browser/projector mode for the reveal. No HTML export is required.

Collect the selected PNGs under `Group_A`, `Group_B`, etc. Embedded prompts are shown when available. Ordinary PNGs stay visible, but prompts that were never recorded cannot be reconstructed; you may optionally supply known prompts in the dictionary below.

Keep the `telephone_*.py` helper files next to this notebook. No API account or image generation needed. Keep practice and live files in separate collections.

In [ ]:
from pathlib import Path
from IPython.display import HTML, FileLink, display
from telephone_gallery import scan, render
from telephone_presentation import presentation

## 1. Choose the downloaded folder

Set `FOLDER` to a directory containing `Group_A`, `Group_B`, etc. For ordinary PNGs, the folder and filename establish their position. Invalid names and duplicate positions are reported.

Leave `FALLBACK_SESSION` and `FALLBACK_MODE` blank to match a uniquely identified embedded session/mode automatically. If all images are ordinary PNGs, you can enter the class session name and `live` or `practice` yourself; otherwise they will appear under `unrecorded`.

If you know a missing prompt, enter it in `PROMPTS`, keyed by the PNG's relative path using forward slashes, e.g. `"Group_A/01.png"`. No prompts are invented.


In [ ]:
FOLDER = ""
FALLBACK_SESSION = ""   # Optional: e.g. "h26_w5" for ordinary PNGs
FALLBACK_MODE = ""      # Optional: "live" or "practice" for ordinary PNGs
PROMPTS = {
    # "Group_A/01.png": "Prompt supplied separately by the student",
    # "Group_A/05_01.png": "Another known prompt",
}
records, warnings = {}, []
if FOLDER:
    records, warnings = scan(
        FOLDER, fallback_session=FALLBACK_SESSION,
        fallback_mode=FALLBACK_MODE, prompt_overrides=PROMPTS,
    )
    entries = [entry for entry in records.values() if entry is not None]
    embedded = sum(e["record"]["record_source"] == "embedded" for e in entries)
    supplied = sum(e["record"]["record_source"] == "manual" for e in entries)
    unavailable = sum(e["record"]["record_source"] == "missing" for e in entries)
    print("Groups found:", sorted({key[2] for key in records}))
    print(f"Images: {len(entries)} | embedded prompts: {embedded} | "
          f"manually supplied prompts: {supplied} | prompts unavailable: {unavailable}")
    for warning in warnings:
        print("Check:", warning)
else:
    print("Set FOLDER to the downloaded class folder, then rerun.")


## 2. Present the chains

Run this cell once to open the interactive presenter. The first image fills the stage; each click on **Next image** reveals the next image in the chain. The **Reveal prompt** control is separate, so you can wait for reactions before showing the text.

After round 5, **Chain summary** shows all five images and prompts together. **Next chain** starts the next sequence. Use the chain dropdown to jump to a particular chain or **Previous** to go back.

Missing images keep their numbered positions. Prompts that are unavailable remain explicitly marked as unavailable.
The image is sized for a laptop or projector by default. Change `IMAGE_HEIGHT` in the next cell (e.g. 320 for a more compact view, 520 for a larger one). The presenter also caps the image to 55% of the browser viewport height.


In [ ]:
IMAGE_HEIGHT = 420  # Maximum image height in pixels (200–1000); also capped at 55% of screen height.
player = presentation(records, image_height=IMAGE_HEIGHT)

## 3. Optional: export the complete gallery

The interactive slideshow is designed for Jupyter. If you also want a single HTML document for later review, set `SAVE_GALLERY = True` below. This exports a static overview of all chains with expandable prompts. A full class may produce a large file.

Do not share the gallery before the game ends: the prompt controls are not access protection.

In [ ]:
SAVE_GALLERY = False
if SAVE_GALLERY and records:
    gallery_html = render(records, warnings)
    target = Path("outputs") / "telephone_reveal.html"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(gallery_html, encoding="utf-8")
    display(FileLink(str(target)))
else:
    print("Use the interactive player above, or set SAVE_GALLERY to True for an HTML overview.")

## If a prompt cannot be revealed

A valid PNG without embedded Telephone data still appears in the sequence. Its prompt is labelled unavailable rather than inferred from the image. If you have the original prompt independently, add it to `PROMPTS` in section 1 and rescan before starting a new presentation.

The export from notebook 02 encodes prompt data in the original PNG pixels. Resizing, screenshots, image editing or a stale export can remove it. Even when present, this is not encryption or access protection.